In [2]:
import cv2
import numpy as np

## Эффекты для видео

In [4]:
input_video = 'rickroll.mp4'
output_effects = 'rickroll_effects.mp4'

cap = cv2.VideoCapture(input_video)

fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
result = cv2.VideoWriter(output_effects, fourcc, fps, (w, h))

frame_num = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    time_sec = frame_num / fps
    processed = frame.copy()
    effect_name = "no effect"

    step = int(time_sec // 5)
    match step:
        case 0:
            pass
        case 1:
            processed = cv2.blur(frame, (3, 3))
            effect_name = "blur 3x3"
        case 2:
            processed = cv2.blur(frame, (5, 5))
            effect_name = "blur 5x5"
        case 3:
            processed = cv2.blur(frame, (9, 9))
            effect_name = "blur 9x9"
        case 4:
            processed = cv2.blur(frame, (15, 15))
            effect_name = "blur 15x15"
        case 5:
            processed[:, :, 0] = 0
            effect_name = "drop Blue"
        case 6:
            processed[:, :, 1] = 0
            effect_name = "drop Green"
        case 7:
            processed[:, :, 2] = 0
            effect_name = "drop Red"
        case 8:
            kernel = np.ones((3, 3), np.uint8)
            processed = cv2.erode(frame, kernel, iterations=1)
            effect_name = "erode 3x3"
        case 9 | 10:
            kernel = np.ones((5, 5), np.uint8)
            processed = cv2.erode(frame, kernel, iterations=1)
            effect_name = "erode 5x5"
        case _:
            break

    cv2.putText(processed, f"{time_sec:.1f}s | {effect_name}", (50, h - 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 3)

    result.write(processed)
    frame_num += 1

cap.release()
result.release()

## Признаки Хаара

In [5]:
input_effects_video = 'rickroll_effects.mp4'
output_faces = 'rickroll_faces.mp4'

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml') #

cap = cv2.VideoCapture(input_effects_video)
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
result = cv2.VideoWriter(output_faces, fourcc, fps, (w, h))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

    for (x, y, w_face, h_face) in faces:
        cv2.rectangle(frame, (x, y), (x + w_face, y + h_face), (0, 255, 0), 3)

    result.write(frame)

cap.release()
result.release()